## Multi-Turn RL (Agentic RFT) Example - Fine-tuning with SageMaker

This notebook demonstrates how to use the `MultiTurnRLTrainer` to fine-tune models using
Multi-Turn Reinforcement Learning (MTRL). MTRL trains models to interact with agent environments
over multiple turns, improving their ability to use tools and follow multi-step reasoning.

**Agent Environment Options:**
- **Bedrock AgentCore**: Deploy your agent on AWS Bedrock AgentCore
- **Your own agent environment**: Use a Lambda function to forward rollouts to your own agent (EKS, Fargate, etc.)

**Sections:**
1. Setup and Configuration
2. List Supported Models
3. Prepare and Register Dataset
4. Create Model Package Groups (optional)
5. Set Up Agent Environment (Bedrock AgentCore or Your own agent environment (EKS, Fargate, etc.) adapted via Lambda)
6. Create MultiTurnRLTrainer and Launch Training
7. Monitor and Inspect Job
8. Attach to Existing Job (optional)
9. List Jobs
10. Continued Training (InputModelPackage)
11. Evaluate Fine-Tuned Model
12. Deploy to SageMaker Endpoint
13. Deploy to Amazon Bedrock

### 1. Setup and Configuration

Initialize the environment by importing necessary libraries and configuring AWS credentials.

In [6]:
import os
import boto3
from sagemaker.core.helper.session_helper import Session

REGION = "us-west-2"
CUSTOMER_ACCOUNT = "123456789012"  # Replace with your account ID

# Configure region
os.environ["AWS_DEFAULT_REGION"] = REGION

# For MLflow native metrics in Trainer wait
os.environ["SAGEMAKER_MLFLOW_CUSTOM_ENDPOINT"] = f"https://mlflow.sagemaker.{REGION}.app.aws"

### 2. List Supported Models

Discover which models support MTRL training.

In [7]:
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer

# Discover all models that support MTRL
models = MultiTurnRLTrainer.list_supported_models()
print(f"Supported models ({len(models)}):")
for m in models:
    print(f"  - {m}")

### 3. Prepare and Register Dataset

Provide your training dataset as an S3 URI or register it in SageMaker AI Registry
for versioning and reuse.

#### Dataset Format Requirements

**Supported file formats:** Apache Parquet (`.parquet`, recommended), JSON Lines (`.jsonl`), JSON (`.json`), CSV (`.csv`)

**Schema:**
- The service auto-detects the prompt column: if a column named `prompt` exists, it is used; otherwise the first column is used.
- Additional columns may be included for tracking but only the prompt column is read by the service.

**How prompts are used:**
- The service is **format-agnostic** — it reads the `prompt` column and passes the string value directly to the agent environment as-is (no parsing or transformation).
- The prompt format depends entirely on what your agent environment expects (plain text, JSON-encoded conversation history, tool-use configs, etc.).

**Data protection:** Since prompts are passed through without inspection, consider encoding (Base64) or encrypting sensitive prompt content. Your agent environment handles decoding/decryption.

**Best practices:**
- Minimum dataset size: at least equal to `global_batch_size` (10x+ recommended for diversity)
- Include complete context in each prompt, maintain consistent structure, avoid duplicates
- For tool-use tasks, provide explicit format instructions in the prompt

**Example — Simple Q&A (Parquet):**
```python
import pyarrow as pa
import pyarrow.parquet as pq

data = {"prompt": ["What is 2 + 2?", "Explain machine learning."]}
table = pa.table(data)
pq.write_table(table, "training_data.parquet")
```

**Example — Multi-turn with tool use (Parquet):**
```python
import json
task_data = {
    "prompt": [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content": "..."}],
    "env_class": "search",
    "reward_spec": {"ground_truth": {"target": ["answer"]}, "style": "rule"},
    "extra_info": {"need_tools_kwargs": True, "tools_kwargs": {...}}
}
data = {"prompt": [json.dumps(task_data)]}
table = pa.table(data)
pq.write_table(table, "search_training_data.parquet")
```

In [ ]:
from sagemaker.ai_registry.dataset import DataSet

# Option A: Use an S3 path directly
S3_TRAINING_DATA = f"s3://{REGION}-{CUSTOMER_ACCOUNT}-{STAGE}-rftjob-input/test-data/prompts/math-100-parquet/"

# Option B: Register a dataset in SageMaker AI Registry
# This creates a versioned dataset that can be referenced by ARN
# dataset = DataSet.create(
#     name="mtrl-training-data",
#     source="s3://your-bucket/training-data/prompts.jsonl"
# )
# print(f"Dataset ARN: {dataset.arn}")
# S3_TRAINING_DATA = dataset.arn

### 4. Create Model Package Groups (optional)

MTRL training uses two Model Package Groups:
- **Output Model Package Group**: Hosts the final trained model package after job completion.
- **Intermediate Checkpoint Model Package Group**: Hosts model checkpoints saved during training, allowing you to resume or evaluate intermediate results.

Both are auto-created if not provided (as `{model}-mtrl-mpg` and `{model}-mtrl-checkpoint-mpg`).
They must be different from each other.

In [ ]:
from sagemaker.core.resources import ModelPackageGroup

# Create Model Package Groups (or use existing ARNs)
# output_mpg = ModelPackageGroup.create(model_package_group_name="my-mtrl-models")
# checkpoint_mpg = ModelPackageGroup.create(model_package_group_name="my-mtrl-checkpoints")
# print(f"Output MPG: {output_mpg.model_package_group_arn}")
# print(f"Checkpoint MPG: {checkpoint_mpg.model_package_group_arn}")

# Or reference existing ones by ARN string:
# OUTPUT_MPG_ARN = "arn:aws:sagemaker:us-west-2:123456789012:model-package-group/my-mtrl-models"
# CHECKPOINT_MPG_ARN = "arn:aws:sagemaker:us-west-2:123456789012:model-package-group/my-mtrl-checkpoints"

### 5. Set Up Agent Environment

Choose one of the following options for your agent environment:
- **Option A**: Bedrock AgentCore (managed agent runtime)
- **Option B**: Your own agent environment (EKS, Fargate, etc.) adapted via Lambda

#### Option A: Bedrock AgentCore

Deploy your agent on Bedrock AgentCore and provide the runtime ARN.

```bash
# Create an agent runtime using the Bedrock AgentCore CLI
aws bedrock-agentcore-control create-agent-runtime \
  --agent-runtime-name my-mtrl-agent \
  --description "Agent for MTRL training" \
  --agent-runtime-artifact '{"containerConfiguration": {"containerUri": "<ECR_IMAGE_URI>"}}'
```

If you have Bedrock AgentCore Runtime deployed in your account, you can find it as following:

In [ ]:
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer
import json

# List all Bedrock Agentcore runtime
agentcore_runtimes = MultiTurnRLTrainer.list_bedrock_agentcore_runtimes()

print(json.dumps(agentcore_runtimes, indent=2))


In [ ]:
# Bedrock AgentCore ARN
AGENT_ENV = "arn:aws:bedrock-agentcore:us-west-2:123456789012:runtime/my-agent-runtime-id"

# Or provide just the runtime ID (will be resolved to full ARN automatically)
# AGENT_ENV = "myRuntime-aBcDeFgHiJ"

#### Option B: Custom Lambda Agent

Use a Lambda function as a forwarder to your custom agent environment.
The Lambda receives rollout requests from the SageMaker service and forwards them
to your agent (deployed on EKS, Fargate, or any HTTP endpoint).

##### Lambda Forwarder Template

Your Lambda forwarder should:
1. Receive rollout requests from SageMaker service
2. Forward them to your agent endpoint
3. Return the agent's response

```python
"""
Lambda Template

Bridges SageMaker Job rollout requests to any agent platform with a public endpoint.
Customers implement `_call_agent()` with their platform-specific logic.

If your agent environment does not have a public endpoint, you can
replace the HTTP call with an SQS send_message to enqueue the request,
and have your agent poll the queue for work.

Env vars:
    AGENT_ENDPOINT  — target agent base URL
    AGENT_API_KEY   — API key for the target agent (prefer Secrets Manager)
"""

import json
import logging
import os
import re
import urllib.error
import urllib.request

logger = logging.getLogger()
logger.setLevel(os.environ.get("LOG_LEVEL", "INFO"))

AGENT_ENDPOINT = os.environ.get("AGENT_ENDPOINT", "")
AGENT_API_KEY = os.environ.get("AGENT_API_KEY", "")

_SAFE_ID = re.compile(r"^[\w\-.]+$")


# ---------------------------------------------------------------------------
# CUSTOMIZE THIS — translate rollout request to your platform's API
# ---------------------------------------------------------------------------
def _call_agent(prompt: str, inference_params: dict) -> dict:
    """
    Forward the prompt to your agent platform and return the response.
    Replace the body below with your platform's request format.
    """
    payload = json.dumps({
        "prompt": prompt,
        "inferenceParams": inference_params,
    }).encode()

    req = urllib.request.Request(
        AGENT_ENDPOINT,
        data=payload,
        headers={
            "Content-Type": "application/json",
            "Authorization": f"Bearer {AGENT_API_KEY}",
        },
        method="POST",
    )

    with urllib.request.urlopen(req, timeout=120) as resp:
        return json.loads(resp.read())


# ---------------------------------------------------------------------------
# Validation & handler — no changes needed below
# ---------------------------------------------------------------------------
def _validate(event: dict) -> dict:
    body = json.loads(event["body"]) if isinstance(event.get("body"), str) else event

    prompt = body.get("prompt")
    if not isinstance(prompt, str) or not prompt.strip():
        raise ValueError("'prompt' is required and must be a non-empty string")

    meta = body.get("metadata")
    if not isinstance(meta, dict):
        raise ValueError("'metadata' is required")
    for key in ("jobId", "experimentId", "rolloutId"):
        val = meta.get(key)
        if not isinstance(val, str) or not _SAFE_ID.match(val):
            raise ValueError(f"metadata.{key} must match [a-zA-Z0-9_\\-.]")

    params = body.get("inferenceParams") or {}
    if not isinstance(params, dict):
        raise ValueError("'inferenceParams' must be an object")

    return {
        "prompt": prompt.strip(),
        "metadata": {k: meta[k] for k in ("jobId", "experimentId", "rolloutId")},
        "inferenceParams": params,
    }


# ---------------------------------------------------------------------------
# CUSTOMIZE THIS — handle errors thrown from your agent environment
# ---------------------------------------------------------------------------
def _handle_agent_error(exc: Exception) -> dict:
    """
    Called when _call_agent() raises an exception.
    Supported errorType values:
        ValidationError, Throttling, InternalServerError, AccessDenied
    """
    logger.exception("Agent environment error")
    if isinstance(exc, urllib.error.HTTPError):
        code = exc.code
        if code == 403:
            return {"errorType": "AccessDenied", "error": "Agent denied access"}
        if code == 429:
            return {"errorType": "Throttling", "error": "Agent rate limit exceeded"}
        if 400 <= code < 500:
            return {"errorType": "ValidationError", "error": str(exc)}
        return {"errorType": "InternalServerError", "error": str(exc)}
    return {"errorType": "InternalServerError", "error": str(exc)}


def handler(event, context):
    try:
        body = _validate(event)
    except ValueError as exc:
        logger.warning("Validation error: %s", exc)
        return {"errorType": "ValidationError", "error": str(exc)}

    try:
        result = _call_agent(body["prompt"], body["inferenceParams"])
        logger.info("Rollout %s completed", body["metadata"]["rolloutId"])
        return {}
    except Exception as exc:
        return _handle_agent_error(exc)
```

In [ ]:
from sagemaker.train.custom_agent_lambda import CustomAgentLambda

# Option 1: Create a new Lambda from inline code or file
adapter = CustomAgentLambda.create(
    source="path/to/lambda_forwarder.py",  # or S3 URI, or inline code string
    function_name="rft-agent-forwarder",
    role="arn:aws:iam::123456789012:role/LambdaForwarderRole",
    timeout=120,
    environment={"AGENT_ENDPOINT": "http://your-agent-loadbalancer-url"},
)

# Option 2: Use an existing Lambda
# adapter = CustomAgentLambda.get("arn:aws:lambda:us-west-2:123456789012:function:rft-agent-forwarder")

# Option 3: Just provide the Lambda ARN string directly
# AGENT_ENV = "arn:aws:lambda:us-west-2:123456789012:function:rft-agent-forwarder"

AGENT_ENV = adapter
print(f"Agent: {adapter}")

### 6. Create MultiTurnRLTrainer and Launch Training

**Required Parameters:**
- `model`: Base model ID on SageMaker Public Hub (e.g., `"openai-reasoning-gpt-oss-20b"`) or a `ModelPackage` object for continued training. Amazon Nova models are also supported, such as `"nova-textgeneration-lite-v2"`.
- `agent_env`: Bedrock AgentCore ARN, agent runtime ID, Lambda ARN, or `CustomAgentLambda` object

**Optional Parameters:**
- `training_dataset`: S3 URI, DataSet ARN, or `DataSet` object for training prompts
- `accept_eula`: Must be `True` to accept the model's end-user license agreement (required for gated models)
- `output_model_package_group`: ModelPackageGroup object or ARN string for the output model (auto-created as `{model}-mtrl-mpg` if not provided)
- `intermediate_checkpoint_model_package_group`: ModelPackageGroup object or ARN string for intermediate checkpoints (auto-created as `{model}-mtrl-checkpoint-mpg` if not provided)
- `mlflow_app_arn`: MLflow App ARN or `MlflowApp` object for experiment tracking (auto-resolved if not provided; requires version >= 3.10)
- `s3_output_path`: S3 path for output artifacts (defaults to `s3://sagemaker-{region}-{account}/output`)
- `validation_dataset`: S3 URI, DataSet ARN, or `DataSet` object for validation data
- `bedrock_agentcore_qualifier`: Qualifier for Bedrock AgentCore runtime (default: `"DEFAULT"`)
- `mlflow_experiment_name`: Custom MLflow experiment name
- `mlflow_run_name`: Custom MLflow run name
- `networking`: `VpcConfig` for the job
- `kms_key_arn`: KMS key ARN for output encryption
- `sagemaker_session`: SageMaker session (uses default if not provided)
- `role`: IAM role ARN (uses default SageMaker execution role if not provided)

In [ ]:
from sagemaker.train.multi_turn_rl_trainer import MultiTurnRLTrainer

trainer = MultiTurnRLTrainer(
    model="openai-reasoning-gpt-oss-20b",
    agent_env=AGENT_ENV,
    training_dataset=S3_TRAINING_DATA,
    s3_output_path="s3://your-bucket/output/",
    accept_eula=True,
    mlflow_experiment_name="test-finetuned-models-exp", 
    mlflow_run_name="test-finetuned-models-run", 
    # output_model_package_group="arn:aws:sagemaker:us-west-2:123456789012:model-package-group/my-group",
    # mlflow_app_arn="arn:aws:sagemaker:us-west-2:123456789012:mlflow-app/my-app",
)

In [ ]:
# View and customize hyperparameters
print("Default Hyperparameters:")
pprint(trainer.hyperparameters.to_dict())

# Optionally modify hyperparameters
# trainer.hyperparameters.num_training_steps = 100
# trainer.hyperparameters.global_batch_size = 16

In [ ]:
# Launch training
job = trainer.train(wait=True)

# Or launch without waiting
# job = trainer.train(wait=False)

### 7. Monitor and Inspect Job

In [ ]:
print(f"Job Name:              {job.job_name}")
print(f"Job ARN:               {job.job_arn}")
print(f"Status:                {job.job_status}")
print(f"Output Model Package:  {job.output_model_package_arn}")
print(f"S3 Output Path:        {job.s3_output_path}")
print(f"Billable Tokens:       {job.billable_token_usage}")
print(f"Progress Info:         {job.progress_info}")

# Get MLflow tracking URL (renders clickable link in Jupyter)
job.get_mlflow_url()

In [ ]:
# View per-step training metrics from MLflow
job.get_training_metrics()

### 8. Attach to an Existing Job

If you have a previously created job, and you want to track the progress, you can use the job name to attach to MultiTurnRLTrainer and use wait() to track the progress again or use other methods to check the info.

In [ ]:
from sagemaker.train.agent_rft_job import AgentRFTJob

# Attach by job name
existing_job = MultiTurnRLTrainer.attach("your-job-name")
# Or: existing_job = AgentRFTJob.get("your-job-name")

print(f"Status: {existing_job.job_status}")
existing_job.get_mlflow_url()

# Wait for completion if still running
if existing_job.job_status == "InProgress":
    existing_job.wait()

### 9. List Jobs

In [ ]:
from sagemaker.train.agent_rft_job import AgentRFTJob

for j in AgentRFTJob.get_all():
    print(f"{j.job_name:50s} {j.job_status:15s}")

### 10. Continued Training (InputModelPackage)

Continue training from a previously fine-tuned model by providing a `ModelPackage` object.
The SDK automatically sets `InputModelPackageArn` in the job config, allowing the training
service to resume from the model's artifacts.

In [ ]:
from sagemaker.core.resources import ModelPackage

# Get the output model package from a previous job
previous_job = AgentRFTJob.get("your-previous-job-name")
previous_model_arn = previous_job.output_model_package_arn
print(f"Previous model: {previous_model_arn}")

# Load the ModelPackage
source_model = ModelPackage.get(model_package_name=previous_model_arn)

# Create trainer with the previous model as input
continued_trainer = MultiTurnRLTrainer(
    model=source_model,  # Pass ModelPackage object for continued training
    agent_env=AGENT_ENV,
    training_dataset=S3_TRAINING_DATA,
    s3_output_path="s3://your-bucket/output/",
    accept_eula=True,
)

# The job config will include InputModelPackageArn automatically
continued_job = continued_trainer.train(wait=True)
print(f"New model: {continued_job.output_model_package_arn}")

### 11. Evaluate Fine-Tuned Model

Evaluate the fine-tuned model against a dataset and agent environment. This measures how well the trained model performs on the task without further training. You can evaluate:
- A fine-tuned model (pass the `trainer` object)
- A base model (pass the model name string) for baseline comparison

In [ ]:
from sagemaker.train.evaluate import MultiTurnRLEvaluator

# Evaluate the fine-tuned model
evaluator = MultiTurnRLEvaluator(
    model=trainer,
    dataset=S3_TRAINING_DATA,
    agent_config=AGENT_ENV,
    s3_output_path="s3://your-bucket/eval-output/",
    role="arn:aws:iam::123456789012:role/SageMakerRole",
)

evaluation = evaluator.evaluate()
print(f"Evaluation started: {evaluation.arn}")

In [ ]:
evaluation.wait()
print(f"Evaluation status: {evaluation.status.overall_status}")

#### Evaluate Base Model (Baseline Comparison)

Evaluate the base model without any fine-tuning to establish a performance baseline.

In [ ]:
from sagemaker.train.evaluate import MultiTurnRLEvaluator

evaluator_base = MultiTurnRLEvaluator(
    model="openai-reasoning-gpt-oss-20b",
    dataset=S3_TRAINING_DATA,
    agent_config=AGENT_ENV,
    s3_output_path="s3://your-bucket/base-eval-output/",
    role="arn:aws:iam::123456789012:role/SageMakerRole",
    accept_eula=True,
)

base_evaluation = evaluator_base.evaluate()
print(f"Base model evaluation started: {base_evaluation.arn}")

In [ ]:
base_evaluation.wait()
print(f"Base model evaluation status: {base_evaluation.status.overall_status}")

#### List All Evaluations

Retrieve all MTRL evaluations in your account to compare results across runs.

In [ ]:
from sagemaker.train.evaluate import MultiTurnRLEvaluator

all_evals = list(MultiTurnRLEvaluator.get_all(region=REGION))
print(f"Total MTRL evaluations: {len(all_evals)}")
for ex in all_evals[:10]:
    print(f"  {ex.name}: {ex.status.overall_status}")

### 12. Deploy to SageMaker Endpoint

Deploy the fine-tuned model to a SageMaker real-time inference endpoint using `ModelBuilder`. After deployment, you can list inference components and invoke the endpoint with chat-completion-style requests.

In [ ]:
import uuid
from sagemaker.serve import ModelBuilder

model_builder = ModelBuilder(model=trainer, instance_type="ml.g6e.48xlarge")
model_builder.build()

endpoint = model_builder.deploy(
    endpoint_name=f"mtrl-endpoint-{uuid.uuid4().hex[:8]}",
    instance_type="ml.g6e.48xlarge",
    initial_instance_count=1,
)

#### List Inference Components

Each endpoint may host one or more inference components. List them to confirm deployment is ready before invoking.

In [ ]:
import boto3

sm_client = boto3.client("sagemaker", region_name=REGION)

inference_components = sm_client.list_inference_components(
    EndpointNameEquals=endpoint.endpoint_name
)["InferenceComponents"]

print(f"Inference components for endpoint '{endpoint.endpoint_name}':")
for ic in inference_components:
    print(f"  - {ic['InferenceComponentName']} (Status: {ic['InferenceComponentStatus']})")

#### Invoke Endpoint

In [ ]:
import json

response = endpoint.invoke(
    body=json.dumps({
        "model": "/opt/ml/model",
        "messages": [{"role": "user", "content": "What is 25 * 4?"}],
        "max_tokens": 200,
        "stream": False,
    }).encode("utf-8"),
    content_type="application/json",
)
print(response)

### 13. Deploy to Amazon Bedrock

Deploy the fine-tuned model to Amazon Bedrock as an imported model. The `deploy()` method
creates the import job and polls until complete — when it returns, the model is ready for
on-demand inference.

If you need dedicated throughput, you can optionally call `create_provisioned_throughput()`
as a separate step after deploy.

In [ ]:
from sagemaker.serve.bedrock_model_builder import BedrockModelBuilder

bedrock_builder = BedrockModelBuilder(model=trainer)

# deploy() waits for import to complete. Model is ready for on-demand inference after this.
bedrock_response = bedrock_builder.deploy(
    imported_model_name="mtrl-finetuned-model",
    role_arn="arn:aws:iam::123456789012:role/BedrockRole",
)
print(f"Import complete! Model: {bedrock_response.get('importedModelName')}")
print(f"Status: {bedrock_response.get('status')}")

#### Optional: Create Provisioned Throughput

If you need dedicated capacity with guaranteed throughput, create provisioned throughput.
This is optional — on-demand inference works immediately after `deploy()` returns.

In [ ]:
# Optional: Create provisioned throughput for dedicated capacity
# model_id is automatically inferred from the previous deploy() call.

# pt_result = bedrock_builder.create_provisioned_throughput(
#     provisioned_model_name="mtrl-provisioned",
#     model_units=1,
# )
# print(f"Provisioned throughput ARN: {pt_result.get('provisionedModelArn')}")

#### Invoke Bedrock Model

In [ ]:
import json
import boto3

model_arn = bedrock_response.get("deploymentArn") or bedrock_response.get("modelArn")
bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)

invoke_response = bedrock_runtime.invoke_model(
    modelId=model_arn,
    body=json.dumps({
        "messages": [{"role": "user", "content": "What is 15 * 7?"}],
        "max_tokens": 256,
    }),
    contentType="application/json",
)

result = json.loads(invoke_response["body"].read())
print(json.dumps(result, indent=2))

### 14. Cleanup

Delete endpoints and resources when no longer needed to avoid ongoing charges.

In [ ]:
# Uncomment to delete the SageMaker endpoint
# endpoint.delete()

# Uncomment to delete the Bedrock provisioned throughput
# bedrock_client.delete_provisioned_model_throughput(provisionedModelId=deployment_name)